In [2]:
import pandas as pd
import json


def discogs_to_dict(discogs_path: str) -> dict:
    dataset = {}
    with open(discogs_path, "r") as f:
        for line in f:
            clique = json.loads(line)
            clique_id = clique.pop("clique_id")
            dataset[clique_id] = clique
    return dataset

print("Load discogs")
discogs = discogs_to_dict("data/discogs/Discogs-VI-YT-20240701.jsonl")

print("Load matched")
df = pd.read_csv("data/matched.csv", sep="\t")

min_score = 90
df = df[df.Score >= min_score]


Load discogs
Load matched


# Filter 
Keep videos only, where title **and** artist match! 

In [ ]:
# Filter for only TITLE AND ARTIST matches
grouped = df.groupby(['youtube_id', 'clique_id'])
unique_counts = grouped['discogs_attr'].nunique()
filtered = unique_counts[unique_counts >= 2]

# To avoid one video mapping to multiple cliques
filtered = filtered[filtered.index.duplicated(keep=False) == False]
print("Total:", len(filtered))


Total: 1072094


In [ ]:
metadata = pd.read_json("data/metadata_filtered.jsonl", lines=True, orient='records')
metadata = metadata.loc[metadata.id.isin(filtered.index.get_level_values(0)),:]

new_columns = ["clique_id"] + metadata.columns.tolist()
metadata = pd.merge(
    filtered.to_frame().reset_index().drop("discogs_attr", axis=1),
    metadata,
    left_on="youtube_id",
    right_on="id",
    how="left",
)
metadata = metadata[new_columns]


In [52]:
def time_to_seconds(time_str):
    parts = list(map(int, time_str.split(":")))
    if len(parts) == 3:  # HH:MM:SS
        h, m, s = parts
        return h * 3600 + m * 60 + s
    elif len(parts) == 2:  # MM:SS
        m, s = parts
        return m * 60 + s
    elif len(parts) == 1:  # SS
        return parts[0]
    else:
        raise ValueError(f"Unrecognized time format: {time_str}")
    
metadata["duration_secs"] = metadata["duration"].apply(time_to_seconds)
metadata["duration_mins"] = metadata["duration_secs"] / 60

# Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.displot(metadata, x="duration_mins", bins=100)
plt.xlabel("Duration (minutes)")
plt.ylabel("Number of videos (log scaled)")
plt.yscale('log')  # <-- set y-axis to log scale
plt.show()

In [ ]:
def get_description_str(descriptionSnippet):
    if isinstance(descriptionSnippet, list):
        return " ".join([d["text"] for d in descriptionSnippet]).replace("\n", " ").replace("\r", " ")
    else:
        return descriptionSnippet

metadata["description"] = metadata.descriptionSnippet.apply(get_description_str)


In [51]:
import pandas as pd
from collections import Counter
from itertools import islice
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

# Make sure you've downloaded stopwords & tokenizer models:
# nltk.download('punkt')
# nltk.download('stopwords')

# Combine stopwords from multiple languages
languages = ['english', 'german', 'french', 'spanish', 'italian', 'portuguese']  # extend as needed
stop_words = set()
for lang in languages:
    stop_words.update(stopwords.words(lang))

def ngrams(tokens, n):
    return zip(*(islice(tokens, i, None) for i in range(n)))

def clean_and_tokenize(text):
    tokens = word_tokenize(text.lower())
    return [word for word in tokens if word.isalnum() and word not in stop_words]

def get_top_ngrams(series, n=1, top_k=20):
    all_ngrams = Counter()
    for text in series.dropna().astype(str):
        tokens = clean_and_tokenize(text)
        all_ngrams.update(ngrams(tokens, n))
    return all_ngrams.most_common(top_k)

# Assuming your dataframe is called df
for column in ['title', 'description']:
    print(f"\n==== {column.upper()} ====")
    for n in [1, 2, 3]:
        top = get_top_ngrams(metadata[column], n=n, top_k=15)  # adjust top_k as needed
        print(f"\nTop {n}-grams (stopwords removed):")
        for phrase, count in top:
            joined = " ".join(phrase) if isinstance(phrase, tuple) else phrase
            print(f"{joined}: {count}")



==== TITLE ====

Top 1-grams (stopwords removed):
live: 51075
love: 44132
cover: 41374
blues: 38224
remastered: 25002
lyrics: 20789
version: 17784
little: 16480
guitar: 15225
karaoke: 13693
baby: 13468
time: 13375
video: 12865
heart: 12469
blue: 10928

Top 2-grams (stopwords removed):
karaoke version: 3946
official audio: 3799
elvis presley: 3532
guitar cover: 3504
first time: 3396
music video: 3221
guitar lesson: 3081
bob dylan: 3035
taylor swift: 2952
bass cover: 2644
remastered 2014: 2589
official video: 2467
time hearing: 2232
drum cover: 2045
acoustic cover: 1994

Top 3-grams (stopwords removed):
first time hearing: 2215
official music video: 1857
karaoke version karafun: 1067
karaoke version zoom: 920
version zoom karaoke: 908
nat king cole: 695
first time reaction: 661
guitar lesson tutorial: 647
creedence clearwater revival: 559
official lyric video: 547
jerry lee lewis: 501
townes van zandt: 478
bass cover tabs: 476
bob dylan cover: 398
cha cha cha: 369

==== DESCRIPTION ====